# Gate 2 - Synthetic Validation And Threshold Registry

Mục tiêu notebook:

- Đọc đúng Gate 1 source of truth đã khóa.
- Chạy Gate 2 synthetic validation workflow từ code trong repo.
- Sinh `synthetic_generator_spec.json`, `synthetic_recovery_report.json`, `synthetic_recovery_report.md`, `gate_threshold_registry.json`, và `gate2_summary.json`.
- Kiểm tra rõ rằng real-data substantive phase vẫn chưa được phép chạy.

Giới hạn liêm chính khoa học:

- Gate 2 ở notebook này là synthetic recovery/control validation proxy, không phải kết quả EEG thật.
- Không dùng raw EDF/MAT, không fit real-data model, không xem performance thật.
- Synthetic pass không chứng minh privileged transfer hiệu quả trên dữ liệu thật.
- Gate 2 chỉ đủ điều kiện để sang Gate 2.5 preregistration bundle nếu pass.
- Phase 0.5/1/2/3 real-data vẫn bị chặn cho đến khi Gate 2.5 hợp lệ.

In [ ]:
# ============================================================
# Gate 2 - Cell 1: Mount Drive, khai báo đường dẫn chuẩn
# ============================================================
# Mục tiêu:
# - Mount Google Drive.
# - Khai báo repo, Gate 0 source, Gate 1 source, Gate 2 output.
# - Không đọc trực tiếp participants.tsv hoặc raw dataset.
# ============================================================

from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = Path('/content/eeg-ds004752')
GATE0_RUN = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/gate0/20260417T102811097110Z')
GATE1_RUN = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/gate1/20260418T153918409528Z')
GATE2_ROOT = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/gate2')

print('Repo:', REPO_DIR)
print('Gate 0 source:', GATE0_RUN)
print('Gate 1 source:', GATE1_RUN)
print('Gate 2 output root:', GATE2_ROOT)

for path in [REPO_DIR, GATE0_RUN, GATE1_RUN]:
    print(path, 'exists =', path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Missing required path: {path}')

In [ ]:
# ============================================================
# Gate 2 - Cell 2: Pull code mới nhất và chạy unit tests
# ============================================================
# Mục tiêu:
# - Lấy commit mới nhất có Gate 2 CLI.
# - Chạy toàn bộ unit tests trước khi sinh artifact.
#
# Kỳ vọng:
# - Có commit mới hơn hoặc bằng commit chứa Gate 2 workflow.
# - Tests pass, hiện kỳ vọng tối thiểu là 21 tests OK.
# ============================================================

import os
import base64
import getpass
import subprocess

%cd /content/eeg-ds004752

def run(cmd, cwd=REPO_DIR, check=True):
    print('\n$', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check)

# Repo private có thể cần token khi pull. Nếu git pull thường fail, dùng token qua http.extraHeader.
pull_result = subprocess.run(['git', 'pull'], cwd=REPO_DIR)
if pull_result.returncode != 0:
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        token = getpass.getpass('git pull cần GitHub token. Nhập token: ')
    basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    extra_header = f'Authorization: Basic {basic}'
    run(['git', '-c', f'http.extraHeader={extra_header}', 'pull'])

run(['git', 'log', '--oneline', '-5'])
run(['python', '-m', 'unittest', 'discover', '-s', 'tests'])

In [ ]:
# ============================================================
# Gate 2 - Cell 3: Preflight kiểm tra Gate 1 source of truth
# ============================================================
# Mục tiêu:
# - Đảm bảo Gate 2 chỉ chạy sau Gate 1 đã decision_layer_ready.
# - Đảm bảo Gate 1 không authorize real-data phase.
# - Đảm bảo Gate 1 next_gate đúng là gate2_synthetic_validation.
# ============================================================

import json

required_gate1_files = [
    'gate1_summary.json',
    'gate1_inputs.json',
    'gate1_input_integrity.json',
    'n_eff_statement.json',
    'simulation_registry.json',
    'sesoi_registry.json',
    'influence_rule.json',
    'decision_memo.md',
]

print('================ Gate 1 Artifact Check ================')
for name in required_gate1_files:
    path = GATE1_RUN / name
    print(name, 'exists =', path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Missing Gate 1 artifact: {path}')

gate1_summary = json.loads((GATE1_RUN / 'gate1_summary.json').read_text())
n_eff = json.loads((GATE1_RUN / 'n_eff_statement.json').read_text())
sesoi = json.loads((GATE1_RUN / 'sesoi_registry.json').read_text())
influence = json.loads((GATE1_RUN / 'influence_rule.json').read_text())
simulation = json.loads((GATE1_RUN / 'simulation_registry.json').read_text())

errors = []
if gate1_summary.get('status') != 'gate1_decision_layer_ready':
    errors.append(f"Gate 1 status invalid: {gate1_summary.get('status')}")
if gate1_summary.get('real_data_phase_authorized') is not False:
    errors.append('Gate 1 must not authorize real-data phases')
if gate1_summary.get('next_gate') != 'gate2_synthetic_validation':
    errors.append(f"Gate 1 next_gate invalid: {gate1_summary.get('next_gate')}")
if n_eff.get('n_primary_eligible') != 15:
    errors.append(f"Unexpected N primary eligible: {n_eff.get('n_primary_eligible')}")
if sesoi.get('primary_subject_level_sesoi', {}).get('median_delta_ba_min') != 0.03:
    errors.append('SESOI Delta BA is not 0.03')
if influence.get('influence_ceiling') != 0.40:
    errors.append('Influence ceiling is not 0.40')
if simulation.get('not_a_model_result') is not True:
    errors.append('Gate 1 simulation must be marked not_a_model_result')

if errors:
    print('Gate 2 preflight FAILED:')
    for item in errors:
        print('-', item)
    raise SystemExit('Stop Gate 2: Gate 1 input is not valid.')

print('\nGate 2 preflight PASSED.')
print('Gate 1 status:', gate1_summary['status'])
print('Gate 1 commit:', gate1_summary['git_commit'])
print('N primary eligible:', n_eff['n_primary_eligible'])
print('SESOI Delta BA:', sesoi['primary_subject_level_sesoi']['median_delta_ba_min'])
print('Influence ceiling:', influence['influence_ceiling'])
print('Real-data phase authorized:', gate1_summary['real_data_phase_authorized'])

In [ ]:
# ============================================================
# Gate 2 - Cell 4: Kiểm tra config synthetic validation
# ============================================================
# Mục tiêu:
# - Đọc config Gate 2 trong repo.
# - Kiểm tra các điều kiện tối thiểu theo tài liệu:
#   * >= 20 synthetic subjects.
#   * 2 classes.
#   * 40-80 trials/class/subject.
#   * Có 3 profile teacher: truly_observable, non_observable, nuisance_shared.
# ============================================================

config_path = REPO_DIR / 'configs/gate2/synthetic_validation.json'
if not config_path.exists():
    raise FileNotFoundError(f'Missing Gate 2 config: {config_path}')

gate2_config = json.loads(config_path.read_text())

profiles = set(gate2_config['effect_profiles'].keys())
required_profiles = {'truly_observable', 'non_observable', 'nuisance_shared'}

assert gate2_config['n_subjects'] >= 20
assert len(gate2_config['classes']) == 2
assert 40 <= gate2_config['trials_per_class_per_subject'] <= 80
assert required_profiles.issubset(profiles)

print('================ Gate 2 Config ================')
print('Config:', config_path)
print('Random seed:', gate2_config['random_seed'])
print('Synthetic subjects:', gate2_config['n_subjects'])
print('Trials/class/subject:', gate2_config['trials_per_class_per_subject'])
print('Classes:', gate2_config['classes'])
print('Profiles:', sorted(profiles))
print('Repeats:', gate2_config['n_repeats'])
print('Frozen defaults:', json.dumps(gate2_config['frozen_threshold_defaults'], indent=2))

In [ ]:
# ============================================================
# Gate 2 - Cell 5: Chạy Gate 2 synthetic validation chính thức
# ============================================================
# Mục tiêu:
# - Chạy CLI từ repo, không tự tính kết quả trong notebook.
# - Sinh artifact Gate 2 vào Google Drive.
#
# Lưu ý:
# - Đây là synthetic proxy, không phải model EEG thật.
# - Nếu synthetic validation fail, không được sang Gate 2.5.
# ============================================================

run([
    'python', '-m', 'src.cli', 'gate2',
    '--profile', 't4_safe',
    '--gate1-run', str(GATE1_RUN),
    '--config', 'configs/gate2/synthetic_validation.json',
    '--output-root', str(GATE2_ROOT),
])

In [ ]:
# ============================================================
# Gate 2 - Cell 6: Đọc và validate gate2_summary.json
# ============================================================
# Mục tiêu:
# - Xác nhận Gate 2 pass synthetic validation.
# - Xác nhận threshold registry được khóa sau synthetic pass.
# - Xác nhận real-data phase vẫn chưa được authorize.
# ============================================================

LATEST_GATE2 = GATE2_ROOT / 'latest.txt'
if not LATEST_GATE2.exists():
    raise FileNotFoundError(f'Missing Gate 2 latest pointer: {LATEST_GATE2}')

gate2_run = Path(LATEST_GATE2.read_text().strip())
summary_path = gate2_run / 'gate2_summary.json'
if not summary_path.exists():
    raise FileNotFoundError(f'Missing Gate 2 summary: {summary_path}')

gate2_summary = json.loads(summary_path.read_text())

print('================ Gate 2 Summary ================')
print('Run dir:', gate2_run)
print('Status:', gate2_summary['status'])
print('Gate 0 source:', gate2_summary['gate0_source_of_truth'])
print('Gate 1 source:', gate2_summary['gate1_source_of_truth'])
print('Git commit:', gate2_summary['git_commit'])
print('Generator hash:', gate2_summary['generator_hash_sha256'])
print('Recovery status:', gate2_summary['recovery_status'])
print('Threshold registry status:', gate2_summary['threshold_registry_status'])
print('Real-data phase authorized:', gate2_summary['real_data_phase_authorized'])
print('Next gate:', gate2_summary['next_gate'])

assert gate2_summary['status'] == 'gate2_synthetic_ready'
assert gate2_summary['recovery_status'] == 'passed'
assert gate2_summary['threshold_registry_status'] == 'locked_after_gate2_pass'
assert gate2_summary['real_data_phase_authorized'] is False
assert gate2_summary['next_gate'] == 'gate2_5_preregistration_bundle'

print('\nGate 2 summary validation PASSED.')

In [ ]:
# ============================================================
# Gate 2 - Cell 7: Kiểm tra artifact bắt buộc
# ============================================================
# Mục tiêu:
# - Đảm bảo mọi artifact Gate 2 bắt buộc có mặt.
# - Không thay thế PDF chính thức; hiện report là Markdown để audit được trong repo/Colab.
# ============================================================

required_gate2_files = [
    'synthetic_generator_spec.json',
    'synthetic_recovery_report.json',
    'synthetic_recovery_report.md',
    'gate_threshold_registry.json',
    'gate2_summary.json',
]

print('================ Gate 2 Artifacts ================')
for name in required_gate2_files:
    path = gate2_run / name
    print(name, 'exists =', path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Missing Gate 2 artifact: {path}')

In [ ]:
# ============================================================
# Gate 2 - Cell 8: Kiểm tra synthetic recovery profiles
# ============================================================
# Mục tiêu:
# - Kiểm tra expected patterns:
#   * truly_observable pass với A4 > A3 > A2.
#   * non_observable không tạo privileged gain robust.
#   * nuisance_shared bị nuisance veto.
#   * shuffled/time-shifted controls pass.
# ============================================================

recovery = json.loads((gate2_run / 'synthetic_recovery_report.json').read_text())

print('================ Synthetic Recovery ================')
print('Status:', recovery['status'])
print('Not real data result:', recovery['not_a_real_data_result'])
print('Generator hash:', recovery['generator_hash_sha256'])
print('Interpretation:', recovery['interpretation'])

assert recovery['status'] == 'passed'
assert recovery['not_a_real_data_result'] is True

for item in recovery['profiles']:
    print('\nProfile:', item['profile'])
    print('  status:', item['status'])
    print('  median A2:', item['median_a2_ba'])
    print('  median A3:', item['median_a3_ba'])
    print('  median A4:', item['median_a4_ba'])
    print('  median A3-A2:', item['median_a3_minus_a2'])
    print('  median A4-A3:', item['median_a4_minus_a3'])
    print('  negative controls passed:', item['negative_controls_passed'])
    print('  nuisance veto applied:', item['nuisance_veto_applied'])
    print('  reason:', item['reason'])
    assert item['status'] == 'passed'
    assert item['negative_controls_passed'] is True

print('\nSynthetic recovery profile validation PASSED.')

In [ ]:
# ============================================================
# Gate 2 - Cell 9: Kiểm tra threshold registry
# ============================================================
# Mục tiêu:
# - Đảm bảo threshold registry đã khóa sau Gate 2 pass.
# - Xác nhận downstream phases phải đọc thresholds từ JSON này.
# ============================================================

threshold_registry = json.loads((gate2_run / 'gate_threshold_registry.json').read_text())

print('================ Gate Threshold Registry ================')
print('Status:', threshold_registry['status'])
print('Recovery status:', threshold_registry['recovery_status'])
print('Registry hash:', threshold_registry['threshold_registry_hash_sha256'])
print('Real-data phase authorized:', threshold_registry['real_data_phase_authorized'])
print('Next gate:', threshold_registry['next_gate'])

print('\nThresholds:')
for key, value in threshold_registry['thresholds'].items():
    print(f'- {key}: {value}')

print('\nProvenance:', threshold_registry['provenance'])

assert threshold_registry['status'] == 'locked_after_gate2_pass'
assert threshold_registry['recovery_status'] == 'passed'
assert threshold_registry['real_data_phase_authorized'] is False
assert threshold_registry['next_gate'] == 'gate2_5_preregistration_bundle'
assert threshold_registry['thresholds']['delta_obs_min'] == 0.02
assert threshold_registry['thresholds']['influence_ceiling'] == 0.40

print('\nThreshold registry validation PASSED.')

In [ ]:
# ============================================================
# Gate 2 - Cell 10: Preview recovery report và kết luận
# ============================================================
# Mục tiêu:
# - Preview report human-readable.
# - In kết luận đúng phạm vi, không overclaim.
# ============================================================

report_text = (gate2_run / 'synthetic_recovery_report.md').read_text()

print('================ Recovery Report Preview ================')
print(report_text[:4000])

print('\n================ Interpretation ================')
print('OK: Gate 2 synthetic validation proxy passed and threshold registry was locked.')
print('OK: Project can proceed to Gate 2.5 preregistration bundle assembly.')
print('NOT OK TO CLAIM: This does not prove real EEG model efficacy or privileged transfer on real data.')
print('BLOCKED: Phase 0.5/1/2/3 real-data substantive runs remain blocked until Gate 2.5 prereg bundle is valid and locked.')
print('\nGate 2 source of truth:', gate2_run)